In [13]:
import json

file_path = "cached_article_database.jsonl"

# Read and store JSONL data
articles = []
with open(file_path, "r", encoding="utf-8") as file:
    for i, line in enumerate(file):
        data = json.loads(line)
        articles.append(data)

        # Preview only the first 5 lines
        if i < 4:
            print(f"Line {i+1}: {data}")

print(f"\n✅ Loaded {len(articles)} articles in total.")

Line 1: {'title': 'Mitsubishi Chemical expands production capacities', 'paragraphs': {'p1': 'Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.', 'p2': 'The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.', 'p3': 'Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com', 'p4': 'Your email address will not be published.Required fields are marked*', 'p5': 'Name *', 'p6': 'E-Mail *', 'p7': 'I agree with thePrivacy policy', 'p8': "We have been covering the development of electric mobility with journalistic passion and expertise since 201

In [14]:
import pandas as pd
from IPython.display import display

# Convert to Pandas DataFrame for better visualization
df = pd.DataFrame(articles)

# Display in notebook
display(df)


,title,paragraphs,meta
0,Mitsubishi Chemical expands production capacities,{'p1': 'Tokyo-based Mitsubishi Chemical wants ...,"{'ID': '1100001', 'date': '04-12-2017', 'url':..."
1,BYD tackles renewables – new battery strategy,{'p1': 'The Chinese EV maker BYD wants to incr...,"{'ID': '1100006', 'date': '05-03-2018', 'url':..."
2,BASF and Toda found cathode subsidiary in USA,{'p1': 'The announced founding of a new subsid...,"{'ID': '1100009', 'date': '10-03-2018', 'url':..."
3,Forsee Power to boost up battery production ca...,{'p1': 'The French battery specialist raises 5...,"{'ID': '1100010', 'date': '02-01-2018', 'url':..."
4,Northvolt secures support from European Invest...,{'p1': 'Northvolt has secured financing of up ...,"{'ID': '1100013', 'date': '13-02-2018', 'url':..."
...,...,...,...
38305,Sekisui Chemical launches pilot perovskite PV ...,{'p1': 'Japanese plastics manufacturer Sekisui...,"{'ID': '1607838', 'date': '06-01-2025', 'url':..."
38306,Waaree Energies starts trial production at 5.4...,{'p1': 'Waaree Energies has started trial prod...,"{'ID': '1607842', 'date': '07-01-2025', 'url':..."
38307,California winery installs solar with dual-axi...,{'p1': 'A winery California will cover 95% of ...,"{'ID': '1607846', 'date': '07-01-2025', 'url':..."
38308,Firefighter injured in solar-plus-storage expl...,"{'p1': 'The fire, which affected a photovoltai...","{'ID': '1607847', 'date': '10-01-2025', 'url':..."


In [16]:
processed_articles = []
for raw_data in articles:
    paragraphs = [raw_data[key] for key in sorted(raw_data.keys()) if key.startswith("p")]

        #ectract the meta-data object
    meta_data = raw_data.get("meta", {})

    # Transform data to fit the new schema
    article = {
        "title": raw_data.get("title", ""),
        "paragraphs": paragraphs,  # Store paragraphs as a list
        "meta": {
            "ID": meta_data.get("ID", ""),
            "date": meta_data.get("date", ""),
            "url": meta_data.get("url", "")
        }
    }

    processed_articles.append(article)

# Display sample processed articles
display(pd.DataFrame(processed_articles).head())

,title,paragraphs,meta
0,Mitsubishi Chemical expands production capacities,[{'p1': 'Tokyo-based Mitsubishi Chemical wants...,"{'ID': '1100001', 'date': '04-12-2017', 'url':..."
1,BYD tackles renewables – new battery strategy,[{'p1': 'The Chinese EV maker BYD wants to inc...,"{'ID': '1100006', 'date': '05-03-2018', 'url':..."
2,BASF and Toda found cathode subsidiary in USA,[{'p1': 'The announced founding of a new subsi...,"{'ID': '1100009', 'date': '10-03-2018', 'url':..."
3,Forsee Power to boost up battery production ca...,[{'p1': 'The French battery specialist raises ...,"{'ID': '1100010', 'date': '02-01-2018', 'url':..."
4,Northvolt secures support from European Invest...,[{'p1': 'Northvolt has secured financing of up...,"{'ID': '1100013', 'date': '13-02-2018', 'url':..."


In [17]:
if processed_articles:
    print(json.dumps(processed_articles[0], indent=4))


{
    "title": "Mitsubishi Chemical expands production capacities",
    "paragraphs": [
        {
            "p1": "Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.",
            "p2": "The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.",
            "p3": "Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com",
            "p4": "Your email address will not be published.Required fields are marked*",
            "p5": "Name *",
            "p6": "E-Mail *",
            "p7": "I agree with thePrivacy policy",
            "p8": "

In [18]:
print(f"Total articles to process: {len(articles)}")
print(f"Total processed articles: {len(processed_articles)}")


Total articles to process: 38310
Total processed articles: 38310


In [19]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]

    # Verify connection
    client.admin.command('ping')
    print("✅ Connected to MongoDB Atlas!")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas!


In [ ]:
import time

total_records = len(processed_articles)
success_count = 0
failures = []

print("\n🚀 Starting data insertion into MongoDB...\n")

for i, article in enumerate(processed_articles, start=1):
    try:
        collection.insert_one(article)
        success_count += 1
        print(f"✅ Processing {i} of {total_records}... Inserted successfully.")
    except Exception as e:
        print(f"❌ Error inserting record {i}: {e}")
        failures.append({"index": i, "error": str(e)})
    time.sleep(0.1)  # Small delay to avoid overwhelming MongoDB (optional)

# Final summary
print(f"\n✅ Successfully inserted {success_count} out of {total_records} records into MongoDB Atlas.")
if failures:
    print(f"⚠️ {len(failures)} records failed to insert. Check error logs.")

# Optional: Save failed records to a log file
if failures:
    with open("mongo_insert_errors.log", "w") as log_file:
        for failure in failures:
            log_file.write(f"Record {failure['index']} - Error: {failure['error']}\n")
    print("⚠️ Failed records have been logged in 'mongo_insert_errors.log'.")


🚀 Starting data insertion into MongoDB...

✅ Processing 1 of 38310... Inserted successfully.
✅ Processing 2 of 38310... Inserted successfully.
✅ Processing 3 of 38310... Inserted successfully.
✅ Processing 4 of 38310... Inserted successfully.
✅ Processing 5 of 38310... Inserted successfully.
✅ Processing 6 of 38310... Inserted successfully.
✅ Processing 7 of 38310... Inserted successfully.
✅ Processing 8 of 38310... Inserted successfully.
✅ Processing 9 of 38310... Inserted successfully.
✅ Processing 10 of 38310... Inserted successfully.
✅ Processing 11 of 38310... Inserted successfully.
✅ Processing 12 of 38310... Inserted successfully.
✅ Processing 13 of 38310... Inserted successfully.
✅ Processing 14 of 38310... Inserted successfully.
✅ Processing 15 of 38310... Inserted successfully.
✅ Processing 16 of 38310... Inserted successfully.
✅ Processing 17 of 38310... Inserted successfully.
✅ Processing 18 of 38310... Inserted successfully.
✅ Processing 19 of 38310... Inserted successful